# Meta: Baseline Evaluations

In [1]:
# Imports
import os, json, sys
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import numpy as np
import seaborn as sns
import language_tool_python as ltp
from typing import List, Dict, Any

In [2]:
# Paths
notebook_path = os.getcwd()
project_root = os.path.dirname(os.path.abspath(notebook_path))
data_dir = os.path.join(project_root, "data")
evaluations_dir = os.path.join(data_dir, "evaluations")
responses_dir = os.path.join(data_dir, "generated")
vocab_dir = os.path.join(data_dir, "vocab")
db_path = os.path.join(data_dir, "DB.db")


In [3]:
# SQL Alchemy session
engine = create_engine(f"sqlite:///{db_path}")
session_factory = sessionmaker(bind=engine)
session = session_factory()

In [4]:
# Helper functions
def get_prompt_eval_dir(prompt_id):
    return os.path.join(evaluations_dir, f"prompt{prompt_id}")

def get_unique_ids(prompt_eval_dir):
    unique_ids = []
    for filename in os.listdir(prompt_eval_dir):
        if filename.endswith(".json"):
            unique_ids.append(filename.split(".json")[0])
    return unique_ids

def load_evaluation(prompt_id):
    prompt_dir = get_prompt_eval_dir(prompt_id)
    unique_ids = get_unique_ids(prompt_dir)
    res = []
    for unique_id in unique_ids:
        file_path = os.path.join(prompt_dir, f"{unique_id}.json")
        with open(file_path, "r", encoding="utf-8") as file:
            res.append(json.load(file))
    return res

def load_evaluations(prompt_ids):
    res_list = []
    for prompt_id in prompt_ids:
        res_list.append(load_evaluation(prompt_id))
    return res_list

In [5]:
# Ground truth
gt_id = "gt.45af4a46-d238-4251-b734-d407602299bd"
gtb_id = "gtb.45af4a46-d258-4251-b734-d407602299bd"

gt_eval_path = os.path.join(evaluations_dir, f"{gt_id}.json")
gtb_eval_path = os.path.join(evaluations_dir, f"{gtb_id}.json")

gt_path = os.path.join(responses_dir, f"{gt_id}.json")
gtb_path = os.path.join(responses_dir, f"{gtb_id}.json")

with open(gt_path, "r", encoding="utf-8") as file:
    gt_res = json.load(file)
with open(gt_path, "r", encoding="utf-8") as file:
    gtb_res = json.load(file)

In [6]:
print(gt_res.keys())

dict_keys(['model', 'prompt', 'unique_id', 'responses'])


In [7]:
def get_whitelist(vocab_dir: str):
    """Get the whitelist from the vocab directory."""
    with open(os.path.join(vocab_dir, "fp.json"), "r", encoding="utf-8") as file:
        whitelist = json.load(file)
    return whitelist

In [8]:
lang_tool = ltp.LanguageTool(language='de-De', remote_server='localhost:8010')
lang_tool.enabled_rules_only = True
lang_tool.enabled_categories = {"GRAMMAR"}

In [10]:
matches_list = []
for response in gt_res["responses"]:
    matches = lang_tool.check(response["raw"]["response"])
    matches_list.append(matches)

## Whole Reports

## Befunds